# Orange dataset: EDA and data checks

This dataset describes orange conditions (`blackspot`, `canker`, `fresh`, `grenning`), not avocado ripeness. The source train/test split is retained. The next notebooks create a stratified validation set from train and keep test held out.

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset import discover_images

DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
train_samples = discover_images(DATA_ROOT / 'train')
test_samples = discover_images(DATA_ROOT / 'test')
class_names = sorted({label for _, label in train_samples})
assert class_names == sorted({label for _, label in test_samples}), 'Train/test classes differ.'
print('Project root:', PROJECT_ROOT)
print('Classes:', class_names)
print('Train images:', len(train_samples), '| Test images:', len(test_samples))

In [ ]:
count_rows = []
for split_name, samples in [('train', train_samples), ('test', test_samples)]:
    counts = pd.Series([label for _, label in samples]).value_counts()
    for class_name in class_names:
        count_rows.append({'split': split_name, 'class': class_name, 'images': int(counts.get(class_name, 0))})
class_counts = pd.DataFrame(count_rows)
display(class_counts.pivot(index='class', columns='split', values='images'))
sns.barplot(data=class_counts, x='class', y='images', hue='split')
plt.title('Orange image counts by class and split')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'class_distribution.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
quality_rows = []
for split_name, samples in [('train', train_samples), ('test', test_samples)]:
    for image_path, label in samples:
        image = cv2.imread(str(image_path))
        if image is None:
            quality_rows.append({'split': split_name, 'class': label, 'path': str(image_path), 'width': None, 'height': None, 'brightness': None, 'read_error': True})
            continue
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        height, width = gray.shape
        quality_rows.append({'split': split_name, 'class': label, 'path': str(image_path), 'width': width, 'height': height, 'brightness': float(gray.mean()), 'read_error': False})

quality = pd.DataFrame(quality_rows)
print('Unreadable files:', int(quality['read_error'].sum()))
display(quality.groupby(['split', 'class'])[['width', 'height', 'brightness']].agg(['min', 'median', 'max']).round(1))
sns.boxplot(data=quality.dropna(), x='class', y='brightness', hue='split')
plt.title('Grayscale brightness by class (lighting variation check)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'brightness_by_class.png', dpi=160, bbox_inches='tight')
plt.show()

if quality['read_error'].any():
    display(quality.loc[quality['read_error'], ['path', 'class', 'split']])

## Findings to record

The source data is imbalanced: `grenning` is the largest class, while the supplied test set contains only 22–33 examples per class. Review the brightness plot and sample images for background, shadow, glare, and label ambiguity. Do not remove images automatically based on brightness alone; investigate examples and document any exclusions. The small test set makes reported metrics uncertain.